In [1]:
import pandas as pd
import numpy as np
import pyarrow as pa

df_tripdata = pd.read_parquet('/Users/tieukbinh/Desktop/de_zoomcamp/homework/green_tripdata_2025-11.parquet')
df_taxi_zone = pd.read_csv('/Users/tieukbinh/Desktop/de_zoomcamp/homework/taxi_zone_lookup.csv')


In [2]:


df_tripdata.dtypes



VendorID                          int32
lpep_pickup_datetime     datetime64[us]
lpep_dropoff_datetime    datetime64[us]
store_and_fwd_flag                  str
RatecodeID                      float64
PULocationID                      int32
DOLocationID                      int32
passenger_count                 float64
trip_distance                   float64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
ehail_fee                       float64
improvement_surcharge           float64
total_amount                    float64
payment_type                    float64
trip_type                       float64
congestion_surcharge            float64
cbd_congestion_fee              float64
dtype: object

In [3]:
df_tripdata[['lpep_pickup_datetime', 'trip_distance']].head()

,lpep_pickup_datetime,trip_distance
0,2025-11-01 00:34:48,0.74
1,2025-11-01 00:18:52,0.95
2,2025-11-01 01:03:14,2.19
3,2025-11-01 00:10:57,5.44
4,2025-11-01 00:03:48,3.20


In [4]:
mask_date = (df_tripdata['lpep_pickup_datetime'] >= '2025-11-01') & (df_tripdata['lpep_pickup_datetime'] < '2025-12-01')
mask_distance = df_tripdata['trip_distance'] <= 1
combined_mask = mask_date & mask_distance
df_counted = df_tripdata.loc[combined_mask].shape
df_counted


(8007, 21)

In [5]:
mask_miles = df_tripdata['trip_distance'] <= 100
distance_value = df_tripdata.loc[mask_miles, 'trip_distance'].max()
df_tripdata[['lpep_pickup_datetime', 'trip_distance']].loc[df_tripdata['trip_distance'] == distance_value]

,lpep_pickup_datetime,trip_distance
18867,2025-11-14 15:36:27,88.03


In [6]:
index_max_distance = df_tripdata.loc[mask_miles, 'trip_distance'].idxmax()
df_tripdata[['lpep_pickup_datetime', 'trip_distance']].loc[index_max_distance]

lpep_pickup_datetime    2025-11-14 15:36:27
trip_distance                         88.03
Name: 18867, dtype: object

In [7]:
df_tripdata.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,2,2025-11-01 00:34:48,2025-11-01 00:41:39,N,1.0,74,42,1.0,0.74,7.2,...,0.5,1.94,0.0,NaN,1.0,11.64,1.0,1.0,0.00,0.0
1,2,2025-11-01 00:18:52,2025-11-01 00:24:27,N,1.0,74,42,2.0,0.95,7.2,...,0.5,0.00,0.0,NaN,1.0,9.70,2.0,1.0,0.00,0.0
2,2,2025-11-01 01:03:14,2025-11-01 01:15:24,N,1.0,83,160,1.0,2.19,13.5,...,0.5,5.00,0.0,NaN,1.0,21.00,1.0,1.0,0.00,0.0
3,2,2025-11-01 00:10:57,2025-11-01 00:24:53,N,1.0,166,127,1.0,5.44,24.7,...,0.5,0.50,0.0,NaN,1.0,27.70,1.0,1.0,0.00,0.0
4,1,2025-11-01 00:03:48,2025-11-01 00:19:38,N,1.0,166,262,1.0,3.20,18.4,...,1.5,1.00,0.0,NaN,1.0,24.65,1.0,1.0,2.75,0.0


In [8]:
df_taxi_zone.head()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


question 5:

In [9]:
zones = (
    'East Harlem North',
    'East Harlem South',
    'Morningside Heights',
    'Forest Hills'
)

mask_zone_q5 = df_taxi_zone['Zone'].isin(zones)
df_taxi_zone.loc[mask_zone_q5]

,LocationID,Borough,Zone,service_zone
73,74,Manhattan,East Harlem North,Boro Zone
74,75,Manhattan,East Harlem South,Boro Zone
94,95,Queens,Forest Hills,Boro Zone
165,166,Manhattan,Morningside Heights,Boro Zone


In [10]:
mask_date_question5 = (df_tripdata['lpep_pickup_datetime'] >= '2025-11-18') & (df_tripdata['lpep_pickup_datetime'] < '2025-11-19')
df_tripdata_question5 = df_tripdata.loc[mask_date_question5]
df_tripdata_zone = df_tripdata_question5.groupby('PULocationID')['total_amount'].sum()
df_tripdata_zone





PULocationID
7      343.60
10      87.24
14      53.83
15      39.50
17     110.25
        ...  
261     34.37
262     26.78
263     71.53
264     18.80
265     61.00
Name: total_amount, Length: 137, dtype: float64

In [11]:
index_zone = df_tripdata_zone.idxmax()
df_taxi_zone.loc[df_taxi_zone['LocationID'] == index_zone]

,LocationID,Borough,Zone,service_zone
73,74,Manhattan,East Harlem North,Boro Zone


Question 6:

In [12]:
mask_nov_2025 = (df_tripdata['lpep_pickup_datetime'] >= '2025-11-01') & (df_tripdata['lpep_pickup_datetime'] < '2025-12-01')

zoneid_ehn = df_taxi_zone.loc[df_taxi_zone['Zone'] == 'East Harlem North', 'LocationID'].values[0]
mask_zone_ehn = df_tripdata['PULocationID'] == zoneid_ehn

combined_mask_drop_zone = mask_nov_2025 & mask_zone_ehn

df_drop_zone = df_tripdata.loc[combined_mask_drop_zone].groupby('DOLocationID')['tip_amount'].sum()

df_drop_zone

DOLocationID
1        20.00
3         0.00
4        15.90
7        78.54
13        9.99
        ...   
261      10.80
262    1488.96
263    2403.17
264      27.41
265      74.53
Name: tip_amount, Length: 137, dtype: float64

In [13]:
index_drop_zone = df_drop_zone.idxmax()
df_taxi_zone.loc[df_taxi_zone['LocationID'] == index_drop_zone]

,LocationID,Borough,Zone,service_zone
235,236,Manhattan,Upper East Side North,Yellow Zone


without agregration

In [14]:
df_tip = df_tripdata.loc[combined_mask_drop_zone, ['DOLocationID', 'tip_amount']]

idx_max_tip = df_tip['tip_amount'].idxmax()
location_id_max_tip = df_tip.loc[idx_max_tip, 'DOLocationID']

location_id_max_tip

np.int32(263)

In [15]:
df_taxi_zone.loc[df_taxi_zone['LocationID'] == location_id_max_tip]

,LocationID,Borough,Zone,service_zone
262,263,Manhattan,Yorkville West,Yellow Zone
